# Task 1 — Record Count

In [1]:
library(RSQLite)
library(tidyverse)

# Create database connection
conn <- dbConnect(SQLite(), "bike_sharing.db")

# Download and load all 4 datasets into database tables
url1 <- "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/seoul_bike_sharing.csv"
url2 <- "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/cities_weather_forecast.csv"
url3 <- "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/bike_sharing_systems.csv"
url4 <- "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-RP0321EN-SkillsNetwork/labs/datasets/world_cities.csv"

# Read CSVs
seoul_bike_sharing <- read_csv(url1)
cities_weather_forecast <- read_csv(url2)
bike_sharing_systems <- read_csv(url3)
world_cities <- read_csv(url4)

# Write to SQLite database tables
dbWriteTable(conn, "SEOUL_BIKE_SHARING", seoul_bike_sharing, overwrite=TRUE)
dbWriteTable(conn, "CITIES_WEATHER_FORECAST", cities_weather_forecast, overwrite=TRUE)
dbWriteTable(conn, "BIKE_SHARING_SYSTEMS", bike_sharing_systems, overwrite=TRUE)
dbWriteTable(conn, "WORLD_CITIES", world_cities, overwrite=TRUE)

# Confirm tables created
dbListTables(conn)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 8465 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (4): DATE, SEASONS, HOLIDAY, FUNCTIONING_DAY
dbl (10): RENTED_BIKE_COUNT, HOUR, TEMPERATURE, HUMIDITY, WIND_SPEED, VISIBI...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 160 Columns: 12
── Column specification ──────────────────────────

[1] "BIKE_SHARING_SYSTEMS"    "CITIES_WEATHER_FORECAST"
[3] "SEOUL_BIKE_SHARING"      "WORLD_CITIES"

In [2]:
dbGetQuery(conn, "
  SELECT COUNT(*) AS TOTAL_RECORDS 
  FROM SEOUL_BIKE_SHARING
")

TOTAL_RECORDS
<int>
8465


# Task 2 — Operational Hours

In [3]:
dbGetQuery(conn, "
  SELECT COUNT(*) AS OPERATIONAL_HOURS
  FROM SEOUL_BIKE_SHARING
  WHERE RENTED_BIKE_COUNT > 0
")

OPERATIONAL_HOURS
<int>
8465


# Task 3 — Weather Outlook for Seoul

In [4]:
dbGetQuery(conn, "
  SELECT *
  FROM CITIES_WEATHER_FORECAST
  WHERE CITY = 'Seoul'
  LIMIT 1
")

CITY,WEATHER,VISIBILITY,TEMP,TEMP_MIN,TEMP_MAX,PRESSURE,HUMIDITY,WIND_SPEED,WIND_DEG,SEASON,FORECAST_DATETIME
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
Seoul,Clear,10000,12.32,10.91,12.32,1015,50,2.18,248,Spring,1618574400


# Task 4 — Seasons

In [6]:
dbGetQuery(conn, "
  SELECT DISTINCT SEASONS
  FROM SEOUL_BIKE_SHARING
")

SEASONS
<chr>
Winter
Spring
Summer
Autumn


# Task 5 — Date Range

In [8]:
dbGetQuery(conn, "
  SELECT MIN(DATE) AS FIRST_DATE,
         MAX(DATE) AS LAST_DATE
  FROM SEOUL_BIKE_SHARING
")

FIRST_DATE,LAST_DATE
<chr>,<chr>
01/01/2018,31/12/2017


# Task 6 — All-Time High

In [9]:
dbGetQuery(conn, "
  SELECT DATE, HOUR, RENTED_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  WHERE RENTED_BIKE_COUNT = (
    SELECT MAX(RENTED_BIKE_COUNT) 
    FROM SEOUL_BIKE_SHARING
  )
")

DATE,HOUR,RENTED_BIKE_COUNT
<chr>,<dbl>,<dbl>
19/06/2018,18,3556


# Task 7 — Hourly Popularity by Season

In [10]:
dbGetQuery(conn, "
  SELECT SEASONS,
         HOUR,
         AVG(TEMPERATURE) AS AVG_TEMPERATURE,
         AVG(RENTED_BIKE_COUNT) AS AVG_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS, HOUR
  ORDER BY AVG_BIKE_COUNT DESC
  LIMIT 10
")

SEASONS,HOUR,AVG_TEMPERATURE,AVG_BIKE_COUNT
<chr>,<dbl>,<dbl>,<dbl>
Summer,18,29.38791,2135.141
Autumn,18,16.03185,1983.333
Summer,19,28.27378,1889.250
Summer,20,27.06630,1801.924
Summer,21,26.27826,1754.065
Spring,18,15.97222,1689.311
Summer,22,25.69891,1567.870
Autumn,17,17.27778,1562.877
Summer,17,30.07691,1526.293


# Task 8 — Rental Seasonality

In [11]:
dbGetQuery(conn, "
  SELECT SEASONS,
         AVG(RENTED_BIKE_COUNT) AS AVG_BIKE_COUNT,
         MIN(RENTED_BIKE_COUNT) AS MIN_BIKE_COUNT,
         MAX(RENTED_BIKE_COUNT) AS MAX_BIKE_COUNT,
         SQRT(AVG(RENTED_BIKE_COUNT * RENTED_BIKE_COUNT) - 
              AVG(RENTED_BIKE_COUNT) * AVG(RENTED_BIKE_COUNT)) AS STD_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS
  ORDER BY AVG_BIKE_COUNT DESC
")

SEASONS,AVG_BIKE_COUNT,MIN_BIKE_COUNT,MAX_BIKE_COUNT,STD_BIKE_COUNT
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Summer,1034.0734,9,3556,690.0884
Autumn,924.1105,2,3298,617.3885
Spring,746.2542,2,3251,618.5247
Winter,225.5412,3,937,150.3374


# Task 9 — Weather Seasonality

In [12]:
dbGetQuery(conn, "
  SELECT SEASONS,
         AVG(TEMPERATURE)           AS AVG_TEMPERATURE,
         AVG(HUMIDITY)              AS AVG_HUMIDITY,
         AVG(WIND_SPEED)            AS AVG_WIND_SPEED,
         AVG(VISIBILITY)            AS AVG_VISIBILITY,
         AVG(DEW_POINT_TEMPERATURE) AS AVG_DEW_POINT,
         AVG(SOLAR_RADIATION)       AS AVG_SOLAR_RADIATION,
         AVG(RAINFALL)              AS AVG_RAINFALL,
         AVG(SNOWFALL)              AS AVG_SNOWFALL,
         AVG(RENTED_BIKE_COUNT)     AS AVG_BIKE_COUNT
  FROM SEOUL_BIKE_SHARING
  GROUP BY SEASONS
  ORDER BY AVG_BIKE_COUNT DESC
")

SEASONS,AVG_TEMPERATURE,AVG_HUMIDITY,AVG_WIND_SPEED,AVG_VISIBILITY,AVG_DEW_POINT,AVG_SOLAR_RADIATION,AVG_RAINFALL,AVG_SNOWFALL,AVG_BIKE_COUNT
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Summer,26.587711,64.98143,1.609420,1501.745,18.750136,0.7612545,0.25348732,0.00000000,1034.0734
Autumn,13.821580,59.04491,1.492101,1558.174,5.150594,0.5227827,0.11765617,0.06350026,924.1105
Spring,13.021685,58.75833,1.857778,1240.912,4.091389,0.6803009,0.18694444,0.00000000,746.2542
Winter,-2.540463,49.74491,1.922685,1445.987,-12.416667,0.2981806,0.03282407,0.24750000,225.5412


# Task 10 — Total Bikes & City Info for Seoul

In [13]:
dbGetQuery(conn, "
  SELECT w.CITY,
         w.COUNTRY,
         w.LAT,
         w.LNG,
         w.POPULATION,
         SUM(b.BICYCLES) AS TOTAL_BIKES
  FROM WORLD_CITIES w, BIKE_SHARING_SYSTEMS b
  WHERE w.CITY = b.CITY
  AND w.CITY = 'Seoul'
")

CITY,COUNTRY,LAT,LNG,POPULATION,TOTAL_BIKES
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Seoul,"Korea, South",37.5833,127,21794000,20000


# Task 11 — Cities with Similar Bike Scale to Seoul

In [14]:
dbGetQuery(conn, "
  SELECT w.CITY,
         w.COUNTRY,
         w.LAT,
         w.LNG,
         w.POPULATION,
         SUM(b.BICYCLES) AS TOTAL_BIKES
  FROM WORLD_CITIES w, BIKE_SHARING_SYSTEMS b
  WHERE w.CITY = b.CITY
  GROUP BY w.CITY
  HAVING TOTAL_BIKES BETWEEN 15000 AND 20000
")

CITY,COUNTRY,LAT,LNG,POPULATION,TOTAL_BIKES
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Beijing,China,39.9050,116.3914,19433000,16000
Ningbo,China,29.8750,121.5492,7639000,15000
Seoul,"Korea, South",37.5833,127.0000,21794000,20000
Shanghai,China,31.1667,121.4667,22120000,19165
Weifang,China,36.7167,119.1000,9373000,20000
Zhuzhou,China,27.8407,113.1469,3855609,20000


In [16]:
dbDisconnect(conn)